# LAB 4 — Dịch máy Anh–Việt bằng Seq2Seq và Luong Attention

> **Phiên bản hướng dẫn dành cho sinh viên mới học**  
> Notebook xây dựng hệ thống dịch máy Anh–Việt bằng **BiLSTM Encoder**, **LSTM Decoder** và **Luong Global Attention có Input Feeding** trên dữ liệu PhoMT rút gọn.

## Mục tiêu học tập

Sau bài thực hành, sinh viên có thể:

1. Nhắc lại hạn chế của Basic Seq2Seq khi chỉ sử dụng một context vector cố định.
2. Phân biệt cách tính score `dot`, `general` và `concat` trong Luong Attention.
3. Giải thích điểm khác nhau cơ bản giữa Bahdanau Attention và Luong Attention.
4. Cài đặt global attention có mask để bỏ qua các vị trí `<pad>`.
5. Hiểu vai trò của **attentional hidden state** và kỹ thuật **input feeding**.
6. Phân biệt teacher forcing khi huấn luyện và giải mã tự hồi quy khi suy luận.
7. Huấn luyện, lưu checkpoint, đánh giá và trực quan hóa ma trận attention.

---

### Quy ước của bài toán

- Ngôn ngữ nguồn: tiếng Anh (`english`).
- Ngôn ngữ đích: tiếng Việt (`vietnamese`).
- Mỗi tệp dữ liệu là một danh sách JSON gồm các cặp câu song ngữ.
- Vocabulary chỉ được xây dựng từ **train set** để tránh rò rỉ dữ liệu.
- Dev set dùng để chọn checkpoint; test set chỉ dùng để báo cáo cuối cùng.


## 1. Cài đặt thư viện

Notebook sử dụng:

- `torch`: xây dựng và huấn luyện mô hình Seq2Seq Attention;
- `nltk`: tính BLEU và chrF;
- `matplotlib`: vẽ loss và ma trận attention;
- `tqdm`: hiển thị tiến trình.

Lệnh `%pip` cài thư viện vào đúng môi trường kernel hiện tại. Nếu môi trường đã có đủ thư viện, cell này sẽ chạy rất nhanh.


In [ ]:
%pip install -q nltk matplotlib


## 2. Import, thiết lập seed và chọn thiết bị

### Vì sao cần đặt random seed?

Khởi tạo trọng số, xáo trộn dữ liệu và Dropout đều có yếu tố ngẫu nhiên. Đặt seed giúp kết quả giữa các lần chạy ổn định hơn.

> Tính tái lập tuyệt đối vẫn có thể phụ thuộc phiên bản CUDA, cuDNN, PyTorch và phần cứng.


In [ ]:
from collections import Counter
from functools import partial
from pathlib import Path
import json
import random
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from nltk.translate.bleu_score import SmoothingFunction, corpus_bleu
from nltk.translate.chrf_score import corpus_chrf
from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence,
    pad_sequence,
)
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


def set_seed(seed: int = 42) -> None:
    """Thiết lập seed cho Python, NumPy và PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


SEED = 42
set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"

print(f"PyTorch version: {torch.__version__}")
print(f"Thiết bị đang sử dụng: {DEVICE}")


## 3. Khai báo và kiểm tra đường dẫn dữ liệu

Notebook giả sử thư mục có cấu trúc:

```text
LAB4.ipynb
data/
├── small-train.json
├── small-dev.json
└── small-test.json
```

Mỗi phần tử trong tệp JSON có dạng:

```json
{
  "english": "How are you?",
  "vietnamese": "Bạn khỏe không?"
}
```

Việc kiểm tra đường dẫn ngay từ đầu giúp phát hiện lỗi cấu hình trước khi chạy các bước tốn thời gian.


In [ ]:
DATA_DIR = Path("data")
TRAIN_PATH = DATA_DIR / "small-train.json"
DEV_PATH = DATA_DIR / "small-dev.json"
TEST_PATH = DATA_DIR / "small-test.json"

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_DIR / "seq2seq_luong_attention_best.pt"

required_paths = [TRAIN_PATH, DEV_PATH, TEST_PATH]
missing_paths = [path for path in required_paths if not path.exists()]

if missing_paths:
    missing_text = "\n".join(f"- {path}" for path in missing_paths)
    raise FileNotFoundError(
        "Không tìm thấy các tệp dữ liệu sau:\n"
        f"{missing_text}\n\n"
        "Hãy đặt dữ liệu đúng cấu trúc hoặc thay đổi biến DATA_DIR."
    )

print("Đã tìm thấy đầy đủ dữ liệu:")
for path in required_paths:
    print(f"- {path}")


## 4. Đọc và kiểm tra dữ liệu JSON

Trước khi xây dựng mô hình, cần kiểm tra:

- tệp có chứa một danh sách JSON hay không;
- mỗi mẫu có đủ hai trường `english` và `vietnamese` hay không;
- giá trị của hai trường có phải chuỗi không;
- có câu rỗng hay không.

Kiểm tra dữ liệu sớm giúp lỗi được báo đúng nguyên nhân thay vì xuất hiện muộn trong `Dataset` hoặc mô hình.


In [ ]:
SOURCE_FIELD = "english"
TARGET_FIELD = "vietnamese"


def load_parallel_json(path: Path) -> list[dict[str, str]]:
    """Đọc và kiểm tra một tệp dữ liệu song ngữ dạng JSON."""
    with path.open("r", encoding="utf-8") as file:
        data = json.load(file)

    if not isinstance(data, list):
        raise ValueError(f"{path} phải chứa một danh sách JSON.")

    cleaned_data = []
    for index, item in enumerate(data):
        if not isinstance(item, dict):
            raise ValueError(f"Mẫu {index} trong {path} không phải dictionary.")

        if SOURCE_FIELD not in item or TARGET_FIELD not in item:
            raise KeyError(
                f"Mẫu {index} trong {path} phải có hai trường "
                f"'{SOURCE_FIELD}' và '{TARGET_FIELD}'."
            )

        source_text = item[SOURCE_FIELD]
        target_text = item[TARGET_FIELD]

        if not isinstance(source_text, str) or not isinstance(target_text, str):
            raise TypeError(f"Mẫu {index} trong {path} phải chứa hai chuỗi văn bản.")

        source_text = source_text.strip()
        target_text = target_text.strip()

        if not source_text or not target_text:
            raise ValueError(f"Mẫu {index} trong {path} có câu rỗng.")

        cleaned_data.append(
            {SOURCE_FIELD: source_text, TARGET_FIELD: target_text}
        )

    return cleaned_data


train_data = load_parallel_json(TRAIN_PATH)
dev_data = load_parallel_json(DEV_PATH)
test_data = load_parallel_json(TEST_PATH)

print(f"Train: {len(train_data):,} cặp câu")
print(f"Dev:   {len(dev_data):,} cặp câu")
print(f"Test:  {len(test_data):,} cặp câu")

print("\nVí dụ một cặp câu:")
print("English   :", train_data[0][SOURCE_FIELD])
print("Vietnamese:", train_data[0][TARGET_FIELD])


## 5. Tokenization và xây dựng vocabulary

### Tokenization

Tokenizer cơ bản dưới đây:

1. chuyển văn bản về chữ thường;
2. tách từ và giữ dấu câu thành token riêng.

Ví dụ:

```text
"How are you?" → ["how", "are", "you", "?"]
```

### Các token đặc biệt

- `<pad>`: bổ sung vào cuối câu ngắn trong batch;
- `<bos>`: đánh dấu bắt đầu câu;
- `<eos>`: đánh dấu kết thúc câu;
- `<unk>`: thay cho token ngoài vocabulary.

### Tránh data leakage

Vocabulary chỉ được xây từ **train set**. Token chỉ xuất hiện trong dev/test phải trở thành `<unk>`. Không quét toàn bộ thư mục dữ liệu để tạo vocabulary.


In [ ]:
TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]", flags=re.UNICODE)


def tokenize(text: str) -> list[str]:
    """Tokenizer đơn giản, dùng chung cho tiếng Anh và tiếng Việt."""
    return TOKEN_PATTERN.findall(text.lower().strip())


def detokenize(tokens: list[str]) -> str:
    """Ghép token thành câu và loại khoảng trắng thừa trước dấu câu."""
    text = " ".join(tokens)
    text = re.sub(r"\s+([.,!?;:%\)\]\}])", r"\1", text)
    text = re.sub(r"([\(\[\{])\s+", r"\1", text)
    return text.strip()


class Vocabulary:
    """Ánh xạ hai chiều giữa token và chỉ số nguyên."""

    PAD_TOKEN = "<pad>"
    BOS_TOKEN = "<bos>"
    EOS_TOKEN = "<eos>"
    UNK_TOKEN = "<unk>"

    def __init__(self, min_freq: int = 1) -> None:
        self.min_freq = min_freq
        self.special_tokens = [
            self.PAD_TOKEN,
            self.BOS_TOKEN,
            self.EOS_TOKEN,
            self.UNK_TOKEN,
        ]
        self.itos: list[str] = []
        self.stoi: dict[str, int] = {}

    def build(self, tokenized_sentences: list[list[str]]) -> None:
        counter = Counter(
            token
            for sentence in tokenized_sentences
            for token in sentence
        )

        # Sắp xếp để vocabulary có thứ tự xác định giữa các lần chạy.
        regular_tokens = [
            token
            for token, frequency in sorted(
                counter.items(), key=lambda item: (-item[1], item[0])
            )
            if frequency >= self.min_freq
            and token not in self.special_tokens
        ]

        self.itos = self.special_tokens + regular_tokens
        self.stoi = {token: index for index, token in enumerate(self.itos)}

    @property
    def pad_idx(self) -> int:
        return self.stoi[self.PAD_TOKEN]

    @property
    def bos_idx(self) -> int:
        return self.stoi[self.BOS_TOKEN]

    @property
    def eos_idx(self) -> int:
        return self.stoi[self.EOS_TOKEN]

    @property
    def unk_idx(self) -> int:
        return self.stoi[self.UNK_TOKEN]

    def __len__(self) -> int:
        return len(self.itos)

    def encode(self, text: str, add_special_tokens: bool = True) -> list[int]:
        token_ids = [
            self.stoi.get(token, self.unk_idx)
            for token in tokenize(text)
        ]

        if add_special_tokens:
            return [self.bos_idx, *token_ids, self.eos_idx]
        return token_ids

    def decode(self, token_ids, stop_at_eos: bool = True) -> str:
        tokens = []
        for token_id in token_ids:
            token_id = int(token_id)

            if stop_at_eos and token_id == self.eos_idx:
                break

            if token_id in {self.pad_idx, self.bos_idx}:
                continue

            token = self.itos[token_id] if token_id < len(self.itos) else self.UNK_TOKEN
            tokens.append(token)

        return detokenize(tokens)

    def ids_to_tokens(
        self,
        token_ids,
        include_eos: bool = False,
    ) -> list[str]:
        """Đổi dãy chỉ số thành token, hữu ích khi vẽ attention."""
        tokens = []
        for token_id in token_ids:
            token_id = int(token_id)
            token = self.itos[token_id] if token_id < len(self.itos) else self.UNK_TOKEN

            if token_id == self.pad_idx:
                continue
            if token_id == self.eos_idx:
                if include_eos:
                    tokens.append(token)
                break
            if token_id != self.bos_idx:
                tokens.append(token)
        return tokens

    def to_dict(self) -> dict:
        return {"min_freq": self.min_freq, "itos": self.itos}

    @classmethod
    def from_dict(cls, state: dict):
        vocabulary = cls(min_freq=state["min_freq"])
        vocabulary.itos = list(state["itos"])
        vocabulary.stoi = {
            token: index for index, token in enumerate(vocabulary.itos)
        }
        return vocabulary


MIN_FREQUENCY = 1

source_vocab = Vocabulary(min_freq=MIN_FREQUENCY)
target_vocab = Vocabulary(min_freq=MIN_FREQUENCY)

source_vocab.build([
    tokenize(item[SOURCE_FIELD]) for item in train_data
])
target_vocab.build([
    tokenize(item[TARGET_FIELD]) for item in train_data
])

print(f"Kích thước vocabulary tiếng Anh : {len(source_vocab):,}")
print(f"Kích thước vocabulary tiếng Việt: {len(target_vocab):,}")
print("Các token đặc biệt:", target_vocab.itos[:4])


## 6. Kiểm tra mã hóa và giải mã một câu

Trước khi tạo `Dataset`, nên kiểm tra thủ công:

- câu được tách thành những token nào;
- chỉ số đầu có phải `<bos>` hay không;
- chỉ số cuối có phải `<eos>` hay không;
- giải mã có khôi phục được nội dung chính hay không.

Đây là một kiểm tra nhỏ nhưng giúp phát hiện sớm lỗi vocabulary và tokenizer.


In [ ]:
example_source = train_data[0][SOURCE_FIELD]
example_target = train_data[0][TARGET_FIELD]

source_ids = source_vocab.encode(example_source)
target_ids = target_vocab.encode(example_target)

print("Câu nguồn       :", example_source)
print("Token nguồn     :", tokenize(example_source))
print("Chỉ số nguồn    :", source_ids)
print("Giải mã nguồn   :", source_vocab.decode(source_ids))

print("\nCâu đích        :", example_target)
print("Token đích      :", tokenize(example_target))
print("Chỉ số đích     :", target_ids)
print("Giải mã đích    :", target_vocab.decode(target_ids))

assert source_ids[0] == source_vocab.bos_idx
assert source_ids[-1] == source_vocab.eos_idx
assert target_ids[0] == target_vocab.bos_idx
assert target_ids[-1] == target_vocab.eos_idx


## 7. Xây dựng `Dataset`

`TranslationDataset` chuyển mỗi cặp câu thành:

- `source_ids`: chuỗi chỉ số tiếng Anh;
- `target_ids`: chuỗi chỉ số tiếng Việt;
- văn bản gốc để hiển thị và đánh giá.

Notebook mã hóa dữ liệu một lần trong `__init__`. Cách này đơn giản và phù hợp với bộ dữ liệu nhỏ. Với dữ liệu rất lớn, có thể token hóa theo yêu cầu hoặc lưu dữ liệu đã tiền xử lý.


In [ ]:
class TranslationDataset(Dataset):
    def __init__(
        self,
        data: list[dict[str, str]],
        source_vocab: Vocabulary,
        target_vocab: Vocabulary,
    ) -> None:
        self.samples = []

        for item in data:
            source_ids = torch.tensor(
                source_vocab.encode(item[SOURCE_FIELD]),
                dtype=torch.long,
            )
            target_ids = torch.tensor(
                target_vocab.encode(item[TARGET_FIELD]),
                dtype=torch.long,
            )

            self.samples.append(
                {
                    "source_ids": source_ids,
                    "target_ids": target_ids,
                    "source_text": item[SOURCE_FIELD],
                    "target_text": item[TARGET_FIELD],
                }
            )

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> dict:
        return self.samples[index]


train_dataset = TranslationDataset(train_data, source_vocab, target_vocab)
dev_dataset = TranslationDataset(dev_data, source_vocab, target_vocab)
test_dataset = TranslationDataset(test_data, source_vocab, target_vocab)

sample = train_dataset[0]
print("source_ids:", sample["source_ids"].shape, sample["source_ids"][:10])
print("target_ids:", sample["target_ids"].shape, sample["target_ids"][:10])


## 8. Tạo batch có padding bằng `collate_fn`

Các câu trong cùng batch thường có độ dài khác nhau. `pad_sequence` bổ sung `<pad>` vào cuối câu ngắn.

Ngoài tensor đã padding, encoder cần **độ dài thật** của từng câu. `pack_padded_sequence` dùng độ dài này để BiLSTM không xem padding là nội dung.

Attention cũng cần một **mask nguồn** để trọng số tại các vị trí `<pad>` bằng 0.


In [ ]:
def collate_translation_batch(
    batch: list[dict],
    source_pad_idx: int,
    target_pad_idx: int,
) -> dict[str, torch.Tensor | list[str]]:
    source_sequences = [item["source_ids"] for item in batch]
    target_sequences = [item["target_ids"] for item in batch]

    source_lengths = torch.tensor(
        [len(sequence) for sequence in source_sequences],
        dtype=torch.long,
    )

    padded_sources = pad_sequence(
        source_sequences,
        batch_first=True,
        padding_value=source_pad_idx,
    )
    padded_targets = pad_sequence(
        target_sequences,
        batch_first=True,
        padding_value=target_pad_idx,
    )

    return {
        "source_ids": padded_sources,
        "source_lengths": source_lengths,
        "target_ids": padded_targets,
        "source_texts": [item["source_text"] for item in batch],
        "target_texts": [item["target_text"] for item in batch],
    }


collate_fn = partial(
    collate_translation_batch,
    source_pad_idx=source_vocab.pad_idx,
    target_pad_idx=target_vocab.pad_idx,
)

example_batch = collate_fn([train_dataset[0], train_dataset[1]])
print("source_ids shape    :", example_batch["source_ids"].shape)
print("source_lengths      :", example_batch["source_lengths"])
print("target_ids shape    :", example_batch["target_ids"].shape)


## 9. Tạo `DataLoader`

- Train loader dùng `shuffle=True` để thay đổi thứ tự mẫu sau mỗi epoch.
- Dev và test loader dùng `shuffle=False` để thứ tự ổn định.
- `pin_memory=True` có thể tăng tốc truyền dữ liệu từ CPU sang GPU.
- `num_workers=0` tương thích tốt với môi trường notebook; có thể tăng nếu máy hỗ trợ.


In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 0

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

batch = next(iter(train_loader))
print("Batch source:", batch["source_ids"].shape)
print("Batch target:", batch["target_ids"].shape)
print("Độ dài thật  :", batch["source_lengths"][:10].tolist())


## 10. Khảo sát độ dài câu

Độ dài câu ảnh hưởng trực tiếp đến:

- lượng bộ nhớ và thời gian huấn luyện;
- số bước decoder cần thực hiện;
- kích thước ma trận attention;
- giá trị `MAX_DECODE_LENGTH` khi dịch.

Trong hệ thống lớn, có thể giới hạn độ dài hoặc gom câu có độ dài gần nhau vào cùng batch. Bài này giữ toàn bộ dữ liệu để tập trung vào kiến trúc.


In [ ]:
def describe_lengths(data: list[dict[str, str]], field: str) -> dict[str, float]:
    lengths = np.array([len(tokenize(item[field])) + 2 for item in data])
    return {
        "min": int(lengths.min()),
        "mean": float(lengths.mean()),
        "median": float(np.median(lengths)),
        "p95": float(np.percentile(lengths, 95)),
        "max": int(lengths.max()),
    }


source_stats = describe_lengths(train_data, SOURCE_FIELD)
target_stats = describe_lengths(train_data, TARGET_FIELD)

print("Độ dài câu tiếng Anh :", source_stats)
print("Độ dài câu tiếng Việt:", target_stats)

MAX_DECODE_LENGTH = max(10, int(target_stats["p95"]) + 10)
print(f"MAX_DECODE_LENGTH = {MAX_DECODE_LENGTH}")


## 11. Lý thuyết Luong Attention

### 1. Từ Basic Seq2Seq đến Attention

Basic Seq2Seq yêu cầu encoder nén toàn bộ câu nguồn vào một vector cố định. Với câu dài, vector này dễ trở thành nút thắt thông tin.

Attention cho phép decoder truy cập toàn bộ encoder outputs:

```text
h̄₁, h̄₂, ..., h̄ₛ
```

Tại bước dịch `t`, Luong Attention thường dùng **hidden state hiện tại** của decoder `hₜ` làm query để so sánh với từng encoder output `h̄ᵢ`.

### 2. Ba hàm tính attention score

Luong và cộng sự mô tả ba cách tính score phổ biến:

```text
dot:     score(hₜ, h̄ᵢ) = hₜᵀh̄ᵢ
general: score(hₜ, h̄ᵢ) = hₜᵀWₐh̄ᵢ
concat:  score(hₜ, h̄ᵢ) = vₐᵀ tanh(Wₐ[hₜ; h̄ᵢ])
```

- `dot` nhanh nhất nhưng yêu cầu hai vector có cùng kích thước;
- `general` thêm một phép chiếu tuyến tính và là lựa chọn mặc định trong notebook;
- `concat` linh hoạt hơn nhưng có nhiều tham số hơn.

Sau khi có score:

```text
αₜ = softmax(scoreₜ)
cₜ = Σᵢ αₜᵢh̄ᵢ
h̃ₜ = tanh(Wc[cₜ; hₜ])
```

Trong đó `h̃ₜ` là **attentional hidden state**, được dùng để dự đoán token tiếp theo.

### 3. Global Attention

Notebook dùng **global attention**: ở mỗi bước đích, decoder so sánh với tất cả vị trí nguồn. Đây là phiên bản dễ cài đặt và trực quan hóa.

### 4. Input Feeding

Vector `h̃ₜ` của bước hiện tại được nối vào đầu vào decoder ở bước kế tiếp. Cơ chế này giúp decoder biết lịch sử attention đã sử dụng và tạo sự liên kết giữa các lần căn chỉnh.

### 5. Khác Bahdanau Attention ở đâu?

Trong cách cài đặt điển hình:

- **Bahdanau** tính attention từ hidden state trước khi cập nhật recurrent state;
- **Luong** cập nhật decoder hidden state trước, sau đó mới tính attention;
- Luong thường kết hợp context với hidden state để tạo `h̃ₜ`.

Cả hai đều cần mask padding và đều học thông qua loss dịch máy.

### 6. Attention mask và suy luận

Vị trí `<pad>` không chứa nội dung. Score tại padding phải được đặt thành số âm rất lớn trước `softmax`.

- Huấn luyện: decoder nhận token đích đúng của bước trước — teacher forcing.
- Suy luận: decoder chỉ nhận token do chính mô hình vừa dự đoán.

> Không được đánh giá chất lượng dịch bằng cách đưa toàn bộ câu đích đúng vào decoder.


In [ ]:
class BidirectionalEncoder(nn.Module):
    """BiLSTM encoder trả về toàn bộ output và trạng thái khởi tạo decoder."""

    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int,
        encoder_hidden_size: int,
        encoder_num_layers: int,
        decoder_hidden_size: int,
        pad_idx: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=pad_idx,
        )
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=encoder_hidden_size,
            num_layers=encoder_num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if encoder_num_layers > 1 else 0.0,
        )

        encoder_output_size = 2 * encoder_hidden_size
        self.hidden_bridge = nn.Linear(encoder_output_size, decoder_hidden_size)
        self.cell_bridge = nn.Linear(encoder_output_size, decoder_hidden_size)

    def forward(
        self,
        source_ids: torch.Tensor,
        source_lengths: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        embedded = self.dropout(self.embedding(source_ids))

        packed = pack_padded_sequence(
            embedded,
            lengths=source_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        packed_outputs, (hidden, cell) = self.lstm(packed)
        encoder_outputs, _ = pad_packed_sequence(
            packed_outputs,
            batch_first=True,
            total_length=source_ids.size(1),
        )

        # Hai phần tử cuối là forward/backward của tầng BiLSTM cuối cùng.
        final_hidden = torch.cat((hidden[-2], hidden[-1]), dim=-1)
        final_cell = torch.cat((cell[-2], cell[-1]), dim=-1)

        decoder_hidden = torch.tanh(self.hidden_bridge(final_hidden))
        decoder_cell = torch.tanh(self.cell_bridge(final_cell))

        return encoder_outputs, decoder_hidden, decoder_cell


class LuongGlobalAttention(nn.Module):
    """Luong global attention với ba score: dot, general và concat."""

    VALID_METHODS = {"dot", "general", "concat"}

    def __init__(
        self,
        encoder_output_size: int,
        decoder_hidden_size: int,
        method: str = "general",
        attention_dim: int = 128,
    ) -> None:
        super().__init__()

        if method not in self.VALID_METHODS:
            raise ValueError(
                f"method phải thuộc {sorted(self.VALID_METHODS)}, nhận được: {method!r}"
            )

        if method == "dot" and encoder_output_size != decoder_hidden_size:
            raise ValueError(
                "Luong dot attention yêu cầu encoder_output_size "
                "bằng decoder_hidden_size."
            )

        self.method = method
        self.encoder_output_size = encoder_output_size
        self.decoder_hidden_size = decoder_hidden_size

        if method == "general":
            self.encoder_projection = nn.Linear(
                encoder_output_size,
                decoder_hidden_size,
                bias=False,
            )
        elif method == "concat":
            self.concat_projection = nn.Linear(
                encoder_output_size + decoder_hidden_size,
                attention_dim,
                bias=False,
            )
            self.score_projection = nn.Linear(attention_dim, 1, bias=False)

    def score(
        self,
        decoder_hidden: torch.Tensor,
        encoder_outputs: torch.Tensor,
    ) -> torch.Tensor:
        """Trả về score có shape (batch, source_length)."""
        if self.method == "dot":
            keys = encoder_outputs
            return torch.bmm(
                keys,
                decoder_hidden.unsqueeze(-1),
            ).squeeze(-1)

        if self.method == "general":
            keys = self.encoder_projection(encoder_outputs)
            return torch.bmm(
                keys,
                decoder_hidden.unsqueeze(-1),
            ).squeeze(-1)

        source_length = encoder_outputs.size(1)
        expanded_query = decoder_hidden.unsqueeze(1).expand(
            -1,
            source_length,
            -1,
        )
        energy = torch.tanh(
            self.concat_projection(
                torch.cat((expanded_query, encoder_outputs), dim=-1)
            )
        )
        return self.score_projection(energy).squeeze(-1)

    def forward(
        self,
        decoder_hidden: torch.Tensor,
        encoder_outputs: torch.Tensor,
        source_mask: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        scores = self.score(decoder_hidden, encoder_outputs)

        # Padding không được nhận attention.
        scores = scores.masked_fill(
            ~source_mask,
            torch.finfo(scores.dtype).min,
        )
        attention_weights = torch.softmax(scores, dim=-1)

        context = torch.bmm(
            attention_weights.unsqueeze(1),
            encoder_outputs,
        ).squeeze(1)

        return context, attention_weights


class LuongAttentionDecoder(nn.Module):
    """LSTMCell decoder dùng Luong attention và input feeding."""

    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int,
        encoder_output_size: int,
        decoder_hidden_size: int,
        attention_method: str,
        attention_dim: int,
        pad_idx: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.decoder_hidden_size = decoder_hidden_size

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=pad_idx,
        )
        self.dropout = nn.Dropout(dropout)

        # Input feeding: embedding hiện tại + attentional hidden của bước trước.
        self.lstm_cell = nn.LSTMCell(
            input_size=embedding_dim + decoder_hidden_size,
            hidden_size=decoder_hidden_size,
        )

        self.attention = LuongGlobalAttention(
            encoder_output_size=encoder_output_size,
            decoder_hidden_size=decoder_hidden_size,
            method=attention_method,
            attention_dim=attention_dim,
        )

        self.attentional_projection = nn.Linear(
            decoder_hidden_size + encoder_output_size,
            decoder_hidden_size,
            bias=False,
        )
        self.output_layer = nn.Linear(decoder_hidden_size, vocab_size)

    def forward_step(
        self,
        input_tokens: torch.Tensor,
        hidden: torch.Tensor,
        cell: torch.Tensor,
        previous_attentional: torch.Tensor,
        encoder_outputs: torch.Tensor,
        source_mask: torch.Tensor,
    ) -> tuple[
        torch.Tensor,
        torch.Tensor,
        torch.Tensor,
        torch.Tensor,
        torch.Tensor,
    ]:
        embedded = self.dropout(self.embedding(input_tokens))
        decoder_input = torch.cat((embedded, previous_attentional), dim=-1)

        # Luong attention dùng hidden state hiện tại sau bước cập nhật decoder.
        hidden, cell = self.lstm_cell(decoder_input, (hidden, cell))

        context, attention_weights = self.attention(
            decoder_hidden=hidden,
            encoder_outputs=encoder_outputs,
            source_mask=source_mask,
        )

        attentional_hidden = torch.tanh(
            self.attentional_projection(
                torch.cat((hidden, context), dim=-1)
            )
        )
        attentional_hidden = self.dropout(attentional_hidden)
        logits = self.output_layer(attentional_hidden)

        return logits, hidden, cell, attentional_hidden, attention_weights


class Seq2SeqLuong(nn.Module):
    def __init__(
        self,
        source_vocab_size: int,
        target_vocab_size: int,
        embedding_dim: int,
        encoder_hidden_size: int,
        encoder_num_layers: int,
        decoder_hidden_size: int,
        attention_method: str,
        attention_dim: int,
        source_pad_idx: int,
        target_pad_idx: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.source_pad_idx = source_pad_idx
        self.decoder_hidden_size = decoder_hidden_size
        encoder_output_size = 2 * encoder_hidden_size

        self.encoder = BidirectionalEncoder(
            vocab_size=source_vocab_size,
            embedding_dim=embedding_dim,
            encoder_hidden_size=encoder_hidden_size,
            encoder_num_layers=encoder_num_layers,
            decoder_hidden_size=decoder_hidden_size,
            pad_idx=source_pad_idx,
            dropout=dropout,
        )
        self.decoder = LuongAttentionDecoder(
            vocab_size=target_vocab_size,
            embedding_dim=embedding_dim,
            encoder_output_size=encoder_output_size,
            decoder_hidden_size=decoder_hidden_size,
            attention_method=attention_method,
            attention_dim=attention_dim,
            pad_idx=target_pad_idx,
            dropout=dropout,
        )

    def encode(
        self,
        source_ids: torch.Tensor,
        source_lengths: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        encoder_outputs, hidden, cell = self.encoder(
            source_ids,
            source_lengths,
        )
        source_mask = source_ids.ne(self.source_pad_idx)
        return encoder_outputs, hidden, cell, source_mask

    def forward(
        self,
        source_ids: torch.Tensor,
        source_lengths: torch.Tensor,
        target_ids: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Forward với teacher forcing 100% để huấn luyện/tính dev loss."""
        encoder_outputs, hidden, cell, source_mask = self.encode(
            source_ids,
            source_lengths,
        )

        batch_size = source_ids.size(0)
        previous_attentional = torch.zeros(
            batch_size,
            self.decoder_hidden_size,
            device=source_ids.device,
        )

        decoder_inputs = target_ids[:, :-1]
        step_logits = []
        step_attentions = []

        for time_step in range(decoder_inputs.size(1)):
            (
                logits,
                hidden,
                cell,
                previous_attentional,
                attention_weights,
            ) = self.decoder.forward_step(
                input_tokens=decoder_inputs[:, time_step],
                hidden=hidden,
                cell=cell,
                previous_attentional=previous_attentional,
                encoder_outputs=encoder_outputs,
                source_mask=source_mask,
            )
            step_logits.append(logits)
            step_attentions.append(attention_weights)

        logits = torch.stack(step_logits, dim=1)
        attentions = torch.stack(step_attentions, dim=1)
        return logits, attentions

    @torch.inference_mode()
    def greedy_decode(
        self,
        source_ids: torch.Tensor,
        source_lengths: torch.Tensor,
        bos_idx: int,
        eos_idx: int,
        max_length: int,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Sinh bản dịch và attention chỉ từ câu nguồn."""
        encoder_outputs, hidden, cell, source_mask = self.encode(
            source_ids,
            source_lengths,
        )

        batch_size = source_ids.size(0)
        previous_attentional = torch.zeros(
            batch_size,
            self.decoder_hidden_size,
            device=source_ids.device,
        )
        current_tokens = torch.full(
            (batch_size,),
            fill_value=bos_idx,
            dtype=torch.long,
            device=source_ids.device,
        )
        finished = torch.zeros(
            batch_size,
            dtype=torch.bool,
            device=source_ids.device,
        )

        generated_tokens = []
        attention_history = []

        for _ in range(max_length):
            (
                logits,
                hidden,
                cell,
                previous_attentional,
                attention_weights,
            ) = self.decoder.forward_step(
                input_tokens=current_tokens,
                hidden=hidden,
                cell=cell,
                previous_attentional=previous_attentional,
                encoder_outputs=encoder_outputs,
                source_mask=source_mask,
            )
            next_tokens = logits.argmax(dim=-1)

            generated_tokens.append(next_tokens)
            attention_history.append(attention_weights)

            finished |= next_tokens.eq(eos_idx)
            if finished.all():
                break

            current_tokens = torch.where(
                finished,
                torch.full_like(next_tokens, eos_idx),
                next_tokens,
            )

        if not generated_tokens:
            empty_tokens = torch.empty(
                (batch_size, 0),
                dtype=torch.long,
                device=source_ids.device,
            )
            empty_attention = torch.empty(
                (batch_size, 0, source_ids.size(1)),
                dtype=torch.float,
                device=source_ids.device,
            )
            return empty_tokens, empty_attention

        return (
            torch.stack(generated_tokens, dim=1),
            torch.stack(attention_history, dim=1),
        )


## 12. Chọn mô hình và siêu tham số

Cấu hình mặc định được giữ ở mức vừa phải để phù hợp bài thực hành:

- encoder hai chiều giúp mỗi token nguồn nhận ngữ cảnh từ cả hai phía;
- decoder hidden size bằng hai lần encoder hidden size để dùng được `dot` khi cần;
- `attention_method="general"` cho phép mô hình học phép chiếu trước khi tính tích vô hướng;
- `attention_dim` chỉ được dùng khi chọn phương pháp `concat`;
- Dropout hỗ trợ giảm overfitting.

Có thể thay `attention_method` bằng `"dot"` hoặc `"concat"` để so sánh. Khi dùng `"dot"`, kích thước encoder output và decoder hidden phải bằng nhau.

Nếu thiếu bộ nhớ, hãy giảm `BATCH_SIZE`, `embedding_dim` hoặc `encoder_hidden_size`.


In [ ]:
MODEL_CONFIG = {
    "embedding_dim": 128,
    "encoder_hidden_size": 128,
    "encoder_num_layers": 2,
    "decoder_hidden_size": 256,
    "attention_method": "general",
    "attention_dim": 128,
    "dropout": 0.30,
}

model = Seq2SeqLuong(
    source_vocab_size=len(source_vocab),
    target_vocab_size=len(target_vocab),
    source_pad_idx=source_vocab.pad_idx,
    target_pad_idx=target_vocab.pad_idx,
    **MODEL_CONFIG,
).to(DEVICE)


def count_trainable_parameters(module: nn.Module) -> int:
    return sum(
        parameter.numel()
        for parameter in module.parameters()
        if parameter.requires_grad
    )


print(model)
print(f"\nLuong score method: {MODEL_CONFIG['attention_method']}")
print(f"Số tham số có thể học: {count_trainable_parameters(model):,}")


## 13. Loss function, optimizer và gradient clipping

### Cross-Entropy Loss

Tại mỗi vị trí, mô hình dự đoán phân phối xác suất trên vocabulary tiếng Việt. Token `<pad>` không phải nội dung nên được bỏ qua bằng `ignore_index`.

### Loss theo token

Vì câu có độ dài khác nhau, loss được theo dõi theo công thức:

```text
tổng loss của các token thật / tổng số token thật
```

### Gradient clipping

RNN/LSTM có thể gặp exploding gradient. `clip_grad_norm_` giới hạn norm của gradient trước khi optimizer cập nhật trọng số.


In [ ]:
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
GRADIENT_CLIP = 1.0
NUM_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 3

criterion = nn.CrossEntropyLoss(
    ignore_index=target_vocab.pad_idx,
    reduction="sum",
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

print(optimizer)


## 14. Hàm huấn luyện một epoch

Trong mỗi batch:

1. Chuyển tensor sang thiết bị.
2. Chạy encoder và decoder với teacher forcing.
3. Dịch nhãn sang trái một vị trí: `target[:, 1:]`.
4. Tính loss trên token không phải `<pad>`.
5. Lan truyền ngược, clipping gradient và cập nhật trọng số.

Attention được tính ở từng bước decoder, nhưng không cần một loss riêng: gradient từ loss dịch máy sẽ học các tham số attention.


In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    target_pad_idx: int,
    gradient_clip: float,
) -> float:
    model.train()

    total_loss_sum = 0.0
    total_target_tokens = 0

    progress_bar = tqdm(loader, desc="Training", leave=False)

    for batch in progress_bar:
        source_ids = batch["source_ids"].to(DEVICE, non_blocking=True)
        source_lengths = batch["source_lengths"]
        target_ids = batch["target_ids"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits, _ = model(source_ids, source_lengths, target_ids)
        expected_tokens = target_ids[:, 1:]

        loss_sum = criterion(
            logits.reshape(-1, logits.size(-1)),
            expected_tokens.reshape(-1),
        )
        token_count = expected_tokens.ne(target_pad_idx).sum()
        loss = loss_sum / token_count.clamp_min(1)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), gradient_clip)
        optimizer.step()

        total_loss_sum += loss_sum.item()
        total_target_tokens += token_count.item()

        progress_bar.set_postfix(
            token_loss=f"{total_loss_sum / max(total_target_tokens, 1):.4f}"
        )

    return total_loss_sum / max(total_target_tokens, 1)


## 15. Hàm đánh giá loss và chất lượng bản dịch

Notebook tách hai kiểu đánh giá:

### Dev loss

Dev loss dùng teacher forcing, phù hợp để theo dõi tối ưu và early stopping.

### BLEU, chrF và ROUGE-L

Các metric dịch phải dùng `greedy_decode`: mô hình chỉ nhìn câu tiếng Anh.

- **BLEU**: mức trùng khớp n-gram;
- **chrF**: n-gram ký tự, hữu ích khi tokenization chưa hoàn hảo;
- **ROUGE-L**: dãy con chung dài nhất.

Metric tự động không đánh giá đầy đủ độ đúng nghĩa và tự nhiên. Luôn xem thêm các ví dụ dịch.


In [ ]:
@torch.inference_mode()
def evaluate_token_loss(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    target_pad_idx: int,
) -> float:
    model.eval()

    total_loss_sum = 0.0
    total_target_tokens = 0

    for batch in tqdm(loader, desc="Evaluating loss", leave=False):
        source_ids = batch["source_ids"].to(DEVICE, non_blocking=True)
        source_lengths = batch["source_lengths"]
        target_ids = batch["target_ids"].to(DEVICE, non_blocking=True)

        logits, _ = model(source_ids, source_lengths, target_ids)
        expected_tokens = target_ids[:, 1:]

        loss_sum = criterion(
            logits.reshape(-1, logits.size(-1)),
            expected_tokens.reshape(-1),
        )
        token_count = expected_tokens.ne(target_pad_idx).sum()

        total_loss_sum += loss_sum.item()
        total_target_tokens += token_count.item()

    return total_loss_sum / max(total_target_tokens, 1)


@torch.inference_mode()
def generate_corpus_predictions(
    model: Seq2SeqLuong,
    loader: DataLoader,
    target_vocab: Vocabulary,
    max_decode_length: int,
) -> tuple[list[str], list[str], list[str]]:
    model.eval()

    sources = []
    references = []
    predictions = []

    for batch in tqdm(loader, desc="Generating translations", leave=False):
        source_ids = batch["source_ids"].to(DEVICE, non_blocking=True)
        source_lengths = batch["source_lengths"]

        predicted_ids, _ = model.greedy_decode(
            source_ids=source_ids,
            source_lengths=source_lengths,
            bos_idx=target_vocab.bos_idx,
            eos_idx=target_vocab.eos_idx,
            max_length=max_decode_length,
        )

        sources.extend(batch["source_texts"])
        references.extend(batch["target_texts"])
        predictions.extend(
            target_vocab.decode(sequence.cpu().tolist())
            for sequence in predicted_ids
        )

    return sources, references, predictions


def longest_common_subsequence_length(
    reference_tokens: list[str],
    prediction_tokens: list[str],
) -> int:
    """Tính độ dài dãy con chung dài nhất bằng quy hoạch động."""
    if not reference_tokens or not prediction_tokens:
        return 0

    previous_row = [0] * (len(prediction_tokens) + 1)

    for reference_token in reference_tokens:
        current_row = [0]
        for column, prediction_token in enumerate(prediction_tokens, start=1):
            if reference_token == prediction_token:
                current_row.append(previous_row[column - 1] + 1)
            else:
                current_row.append(max(previous_row[column], current_row[-1]))
        previous_row = current_row

    return previous_row[-1]


def rouge_l_f1(reference: str, prediction: str) -> float:
    """Tính ROUGE-L F1 trên token; kết quả nằm trong khoảng [0, 1]."""
    reference_tokens = tokenize(reference)
    prediction_tokens = tokenize(prediction)

    if not reference_tokens or not prediction_tokens:
        return 0.0

    lcs_length = longest_common_subsequence_length(
        reference_tokens,
        prediction_tokens,
    )
    precision = lcs_length / len(prediction_tokens)
    recall = lcs_length / len(reference_tokens)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


def compute_translation_metrics(
    references: list[str],
    predictions: list[str],
) -> dict[str, float]:
    if len(references) != len(predictions):
        raise ValueError("Số câu tham chiếu và dự đoán phải bằng nhau.")

    tokenized_references = [[tokenize(reference)] for reference in references]
    tokenized_predictions = [tokenize(prediction) for prediction in predictions]

    smoothing = SmoothingFunction().method1
    bleu = 100 * corpus_bleu(
        tokenized_references,
        tokenized_predictions,
        smoothing_function=smoothing,
    )

    chrf = 100 * corpus_chrf(references, predictions)

    rouge_scores = [
        rouge_l_f1(reference, prediction)
        for reference, prediction in zip(references, predictions)
    ]
    rouge_l = 100 * float(np.mean(rouge_scores)) if rouge_scores else 0.0

    return {
        "BLEU": bleu,
        "chrF": chrf,
        "ROUGE-L": rouge_l,
    }


## 16. Huấn luyện và lưu checkpoint tốt nhất

Mô hình được lưu khi **dev loss giảm**.

Checkpoint gồm:

- trọng số mô hình và trạng thái optimizer;
- cấu hình mô hình;
- vocabulary nguồn và đích;
- epoch, lịch sử loss và dev loss tốt nhất.

Early stopping dừng nếu dev loss không cải thiện trong một số epoch liên tiếp.


In [ ]:
history = {
    "train_loss": [],
    "dev_loss": [],
}

best_dev_loss = float("inf")
epochs_without_improvement = 0
training_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        target_pad_idx=target_vocab.pad_idx,
        gradient_clip=GRADIENT_CLIP,
    )

    dev_loss = evaluate_token_loss(
        model=model,
        loader=dev_loader,
        criterion=criterion,
        target_pad_idx=target_vocab.pad_idx,
    )

    history["train_loss"].append(train_loss)
    history["dev_loss"].append(dev_loss)

    improved = dev_loss < best_dev_loss
    marker = " ← lưu checkpoint" if improved else ""

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"Train token loss: {train_loss:.4f} | "
        f"Dev token loss: {dev_loss:.4f}{marker}"
    )

    if improved:
        best_dev_loss = dev_loss
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "best_dev_loss": best_dev_loss,
                "model_config": MODEL_CONFIG,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "source_vocab": source_vocab.to_dict(),
                "target_vocab": target_vocab.to_dict(),
                "history": history,
                "max_decode_length": MAX_DECODE_LENGTH,
            },
            CHECKPOINT_PATH,
        )
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(
            f"Early stopping: dev loss không cải thiện trong "
            f"{EARLY_STOPPING_PATIENCE} epoch liên tiếp."
        )
        break

elapsed = time.time() - training_start
print(f"\nThời gian huấn luyện: {elapsed / 60:.2f} phút")
print(f"Checkpoint tốt nhất: {CHECKPOINT_PATH}")
print(f"Best dev token loss: {best_dev_loss:.4f}")


## 17. Vẽ biểu đồ quá trình học

Biểu đồ train loss và dev loss giúp nhận biết:

- cả hai cùng giảm: mô hình đang học;
- train loss giảm nhưng dev loss tăng: dấu hiệu overfitting;
- cả hai gần như không giảm: cần xem lại learning rate, dữ liệu hoặc kiến trúc.

Không nên chỉ nhìn train loss để kết luận mô hình tốt.


In [ ]:
epoch_axis = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epoch_axis, history["train_loss"], marker="o", label="Train loss")
plt.plot(epoch_axis, history["dev_loss"], marker="o", label="Dev loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss trên token")
plt.title("Quá trình huấn luyện Seq2Seq với Luong Attention")
plt.grid(alpha=0.3)
plt.legend()
plt.show()


## 18. Tải lại checkpoint tốt nhất

Không nên mặc định dùng trọng số của epoch cuối. Epoch tốt nhất trên dev set có thể xuất hiện sớm hơn.

Vocabulary cũng được tải từ checkpoint để bảo đảm chỉ số token khớp hoàn toàn với embedding đã huấn luyện.


In [ ]:
def load_checkpoint(path: Path, device: torch.device) -> dict:
    # Tương thích với phiên bản PyTorch có và chưa có tham số weights_only.
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


checkpoint = load_checkpoint(CHECKPOINT_PATH, DEVICE)

loaded_source_vocab = Vocabulary.from_dict(checkpoint["source_vocab"])
loaded_target_vocab = Vocabulary.from_dict(checkpoint["target_vocab"])

best_model = Seq2SeqLuong(
    source_vocab_size=len(loaded_source_vocab),
    target_vocab_size=len(loaded_target_vocab),
    source_pad_idx=loaded_source_vocab.pad_idx,
    target_pad_idx=loaded_target_vocab.pad_idx,
    **checkpoint["model_config"],
).to(DEVICE)

best_model.load_state_dict(checkpoint["model_state_dict"])
best_model.eval()

print(f"Đã tải checkpoint tại epoch {checkpoint['epoch']}")
print(f"Dev token loss tốt nhất: {checkpoint['best_dev_loss']:.4f}")


## 19. Đánh giá cuối cùng trên test set

Test set chỉ được dùng sau khi đã chọn mô hình bằng dev set.

```text
Train set → học tham số
Dev set   → chọn checkpoint và siêu tham số
Test set  → báo cáo kết quả cuối cùng
```

Không điều chỉnh mô hình nhiều lần dựa trên test set, vì khi đó test set không còn là dữ liệu chưa quan sát.


In [ ]:
test_token_loss = evaluate_token_loss(
    model=best_model,
    loader=test_loader,
    criterion=criterion,
    target_pad_idx=loaded_target_vocab.pad_idx,
)

sources, references, predictions = generate_corpus_predictions(
    model=best_model,
    loader=test_loader,
    target_vocab=loaded_target_vocab,
    max_decode_length=checkpoint["max_decode_length"],
)

test_metrics = compute_translation_metrics(references, predictions)

print(f"Test token loss: {test_token_loss:.4f}")
for metric_name, metric_value in test_metrics.items():
    print(f"{metric_name:8s}: {metric_value:.2f}")


## 20. Xem một số bản dịch

Chỉ số tổng hợp không cho biết mô hình sai như thế nào. Cần xem trực tiếp:

- câu nguồn;
- câu tham chiếu;
- câu mô hình sinh ra.

Các lỗi thường gặp gồm bỏ sót từ, lặp từ, kết thúc quá sớm, chọn sai nghĩa và sinh `<unk>`.


In [ ]:
def show_translation_examples(
    sources: list[str],
    references: list[str],
    predictions: list[str],
    number_of_examples: int = 10,
    seed: int = 42,
) -> None:
    rng = random.Random(seed)
    indices = list(range(len(sources)))
    rng.shuffle(indices)

    for order, index in enumerate(indices[:number_of_examples], start=1):
        print(f"Ví dụ {order}")
        print("EN  :", sources[index])
        print("REF :", references[index])
        print("PRED:", predictions[index] or "<câu rỗng>")
        print("-" * 80)


show_translation_examples(
    sources,
    references,
    predictions,
    number_of_examples=min(10, len(sources)),
)


## 21. Dịch câu mới và trực quan hóa Attention

Quy trình suy luận:

1. Mã hóa câu tiếng Anh.
2. Encoder tạo biểu diễn cho từng token nguồn.
3. Decoder bắt đầu bằng `<bos>` và attentional hidden ban đầu bằng 0.
4. Decoder cập nhật hidden state hiện tại.
5. Luong Attention tạo context vector và attentional hidden state.
6. Attentional hidden được dùng để dự đoán token và làm input feeding cho bước sau.
7. Dừng khi gặp `<eos>` hoặc đạt độ dài tối đa.

Ma trận attention có:

- trục ngang: token tiếng Anh;
- trục dọc: token tiếng Việt được sinh;
- mỗi hàng có tổng gần bằng 1.

> Attention là tín hiệu hỗ trợ diễn giải, nhưng không nên mặc định xem nó là lời giải thích nhân quả hoàn chỉnh cho quyết định của mô hình.


In [ ]:
@torch.inference_mode()
def translate_with_attention(
    sentence: str,
    model: Seq2SeqLuong,
    source_vocab: Vocabulary,
    target_vocab: Vocabulary,
    max_decode_length: int,
) -> tuple[str, list[str], list[str], np.ndarray]:
    model.eval()

    source_id_list = source_vocab.encode(sentence)
    source_tensor = torch.tensor(
        source_id_list,
        dtype=torch.long,
    ).unsqueeze(0).to(DEVICE)
    source_length = torch.tensor([len(source_id_list)], dtype=torch.long)

    predicted_ids, attentions = model.greedy_decode(
        source_ids=source_tensor,
        source_lengths=source_length,
        bos_idx=target_vocab.bos_idx,
        eos_idx=target_vocab.eos_idx,
        max_length=max_decode_length,
    )

    predicted_id_list = predicted_ids[0].cpu().tolist()
    source_tokens = [source_vocab.BOS_TOKEN] + tokenize(sentence) + [source_vocab.EOS_TOKEN]
    target_tokens = target_vocab.ids_to_tokens(
        predicted_id_list,
        include_eos=True,
    )

    # Chỉ giữ số hàng tương ứng với các token đích đang hiển thị.
    attention_matrix = attentions[0, :len(target_tokens), :len(source_tokens)]
    attention_matrix = attention_matrix.cpu().numpy()

    translation = target_vocab.decode(predicted_id_list)
    return translation, source_tokens, target_tokens, attention_matrix


def plot_attention(
    source_tokens: list[str],
    target_tokens: list[str],
    attention_matrix: np.ndarray,
) -> None:
    if not target_tokens or attention_matrix.size == 0:
        print("Không có đủ token để vẽ attention.")
        return

    figure_width = max(8, 0.7 * len(source_tokens))
    figure_height = max(5, 0.55 * len(target_tokens))

    plt.figure(figsize=(figure_width, figure_height))
    plt.imshow(attention_matrix, aspect="auto")
    plt.xticks(range(len(source_tokens)), source_tokens, rotation=45, ha="right")
    plt.yticks(range(len(target_tokens)), target_tokens)
    plt.xlabel("Token nguồn — tiếng Anh")
    plt.ylabel("Token đích — tiếng Việt")
    plt.title("Ma trận Luong Attention")
    plt.colorbar(label="Attention weight")
    plt.tight_layout()
    plt.show()


new_sentence = "Thank you very much."
translation, source_tokens, target_tokens, attention_matrix = translate_with_attention(
    sentence=new_sentence,
    model=best_model,
    source_vocab=loaded_source_vocab,
    target_vocab=loaded_target_vocab,
    max_decode_length=checkpoint["max_decode_length"],
)

print("EN:", new_sentence)
print("VI:", translation or "<câu rỗng>")
print("Tổng attention của từng bước:", attention_matrix.sum(axis=1))

plot_attention(source_tokens, target_tokens, attention_matrix)


## 22. Phân tích hạn chế và hướng cải tiến

Luong Attention khắc phục nút thắt context vector của Basic Seq2Seq, nhưng mô hình vẫn có hạn chế.

### 1. Global attention tốn chi phí theo độ dài câu

Mỗi bước đích so sánh với mọi vị trí nguồn. Chi phí và kích thước ma trận attention tăng theo tích độ dài nguồn và đích.

**Cải tiến:** local attention, giới hạn cửa sổ hoặc kiến trúc tối ưu hơn.

### 2. Giải mã tuần tự

Decoder phải sinh từng token nên huấn luyện và suy luận khó song song hóa.

**Cải tiến:** Transformer cho phép xử lý song song tốt hơn trong huấn luyện.

### 3. Greedy decoding

Token tốt nhất ở từng bước chưa chắc tạo thành câu tốt nhất toàn cục.

**Cải tiến:** Beam Search, length penalty hoặc diverse decoding.

### 4. Vocabulary theo từ

Từ hiếm và từ mới trở thành `<unk>`.

**Cải tiến:** BPE, WordPiece hoặc SentencePiece.

### 5. Exposure bias

Huấn luyện dùng token đúng của bước trước, nhưng suy luận dùng token mô hình tự sinh.

**Cải tiến:** scheduled sampling hoặc các phương pháp huấn luyện theo chuỗi nâng cao.

### 6. Score function cần được kiểm chứng

`dot`, `general` và `concat` có độ phức tạp và số tham số khác nhau. Không có một lựa chọn luôn tốt nhất cho mọi bộ dữ liệu.

**Cải tiến:** huấn luyện các biến thể trong cùng điều kiện và so sánh trên dev set.

### 7. Đánh giá tự động

BLEU, chrF và ROUGE-L không phản ánh đầy đủ độ đúng nghĩa và tự nhiên.

**Cải tiến:** kết hợp đánh giá con người và metric ngữ nghĩa phù hợp.


In [ ]:
# Gợi ý phân tích mối liên hệ giữa độ dài câu và chất lượng dự đoán.
length_error_rows = []

for source, reference, prediction in zip(sources, references, predictions):
    length_error_rows.append(
        {
            "source_length": len(tokenize(source)),
            "target_length": len(tokenize(reference)),
            "prediction_length": len(tokenize(prediction)),
            "rouge_l": rouge_l_f1(reference, prediction),
        }
    )

if length_error_rows:
    sorted_rows = sorted(length_error_rows, key=lambda row: row["rouge_l"])
    print("Năm mẫu có ROUGE-L thấp nhất:")
    for row in sorted_rows[:5]:
        print(row)


## 23. Tổng kết kiến thức

Trong bài thực hành này, sinh viên đã hoàn thành pipeline:

```text
Đọc và kiểm tra dữ liệu song ngữ
        ↓
Tokenization + vocabulary chỉ từ train set
        ↓
Dataset + padding + độ dài thật
        ↓
Packed BiLSTM Encoder
        ↓
LSTM Decoder cập nhật hidden state
        ↓
Luong Global Attention có padding mask
        ↓
Attentional hidden + input feeding
        ↓
Teacher forcing khi huấn luyện
        ↓
Early stopping + checkpoint
        ↓
Autoregressive greedy decoding
        ↓
BLEU + chrF + ROUGE-L
        ↓
Trực quan hóa ma trận attention
```

### Những điểm cần ghi nhớ

1. Luong Attention thường tính score bằng hidden state hiện tại của decoder.
2. Ba score phổ biến là `dot`, `general` và `concat`.
3. Context và decoder hidden được kết hợp để tạo attentional hidden state.
4. Input feeding đưa attentional hidden của bước trước vào decoder ở bước sau.
5. Attention score tại vị trí padding phải bị mask trước softmax.
6. Teacher forcing phù hợp cho huấn luyện/dev loss, không phù hợp để đo chất lượng dịch thực tế.
7. Checkpoint tốt nhất được chọn trên dev set trước khi báo cáo test set.
8. Attention weight hữu ích để quan sát nhưng không phải bằng chứng giải thích nhân quả hoàn chỉnh.

### Bài tập mở rộng

1. So sánh Basic LSTM, Bahdanau Attention và Luong Attention trên cùng dữ liệu.
2. Huấn luyện lần lượt với `dot`, `general`, `concat`; lập bảng số tham số và metric.
3. Tắt input feeding và nhận xét sự thay đổi.
4. Thay greedy decoding bằng Beam Search.
5. Vẽ attention cho câu ngắn, trung bình và dài.
6. Thử subword tokenization và phân tích số lượng `<unk>`.


In [ ]:
print("Hoàn thành LAB 4 — Bài 3: Seq2Seq với Luong Attention.")
print("Bước tiếp theo đề xuất: Beam Search, subword tokenization hoặc Transformer.")
